## 02 - Базовые модели (Baseline)

В этом ноутбуке обучаем простые модели «из коробки» без настройки гиперпараметров.
Цель — установить нижнюю и начальную точки отсчёта для дальнейших экспериментов.

**Модели:**
1. DummyClassifier - нижняя граница
2. LogisticRegression - линейная базовая модель
3. KNeighborsClassifier - простая нелинейная модель

Признаки: аудио + тематические вероятности + текстовые + мета (27 штук)

Метрика оценки: Macro F1 (основная) + Accuracy

## 0. Импорты и настройка

In [2]:
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, "../src")
from preprocessing import get_all_feature_columns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Зависимости загружены")

Зависимости загружены


## 1. Загрузка данных

In [3]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Train: (19860, 35), Val: (4256, 35), Test: (4256, 35)


In [4]:
# Определяем признаки и целевую переменную
feature_cols = get_all_feature_columns()

# Оставляем только те столбцы, которые есть в данных
feature_cols = [c for c in feature_cols if c in train_df.columns]
print(f"Используем {len(feature_cols)} признаков:")
print(feature_cols)

Используем 27 признаков:
['danceability', 'loudness', 'acousticness', 'instrumentalness', 'valence', 'energy', 'dating', 'violence', 'world_life', 'night_time', 'shake_the_audience', 'family_gospel', 'romantic', 'communication', 'obscene', 'music', 'movement_places', 'light_visual_perceptions', 'family_spiritual', 'like_girls', 'sadness', 'feelings', 'repetition_ratio', 'avg_word_length', 'sentiment_polarity', 'len', 'age']


In [5]:
# Подготовка матриц признаков
X_train = train_df[feature_cols].fillna(0).values
y_train = train_df["genre"].values

X_val = val_df[feature_cols].fillna(0).values
y_val = val_df["genre"].values

X_test = test_df[feature_cols].fillna(0).values
y_test = test_df["genre"].values

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"Классы: {sorted(set(y_train))}")

X_train: (19860, 27), X_val: (4256, 27), X_test: (4256, 27)
Классы: ['blues', 'country', 'hip hop', 'jazz', 'pop', 'reggae', 'rock']


## 2. Вспомогательная функция для оценки моделей

In [7]:
results = []  # накапливаем результаты всех моделей


def evaluate_model(model, X_tr, y_tr, X_vl, y_vl, name: str) -> dict:
    """Обучает модель и оценивает на val."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_vl)

    macro_f1 = f1_score(y_vl, y_pred, average="macro", zero_division=0)
    accuracy = accuracy_score(y_vl, y_pred)

    result = {
        "Модель": name,
        "Macro F1 (val)": round(macro_f1, 4),
        "Accuracy (val)": round(accuracy, 4),
    }
    results.append(result)
    print(f"{name}: Macro F1 = {macro_f1:.4f}, Accuracy = {accuracy:.4f}")
    return result

## 3. DummyClassifier — нижняя граница

In [8]:
# DummyClassifier случайно предсказывает класс пропорционально частоте в train.
# Будет считаться нижней границей, другие модели должны быть лучше
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
evaluate_model(dummy, X_train, y_train, X_val, y_val, "DummyClassifier (stratified)")

DummyClassifier (stratified): Macro F1 = 0.1419, Accuracy = 0.1713


{'Модель': 'DummyClassifier (stratified)',
 'Macro F1 (val)': 0.1419,
 'Accuracy (val)': 0.1713}

## 4. Логистическая регрессия — линейная базовая модель

In [9]:
# Логистическая регрессия без настройки гиперпараметров.
logreg_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

evaluate_model(
    logreg_pipeline, X_train, y_train, X_val, y_val, "LogisticRegression (baseline)"
)

LogisticRegression (baseline): Macro F1 = 0.4161, Accuracy = 0.4077


{'Модель': 'LogisticRegression (baseline)',
 'Macro F1 (val)': 0.4161,
 'Accuracy (val)': 0.4077}

In [10]:
# Подробный отчёт по классам
logreg_pipeline.fit(X_train, y_train)
y_pred_logreg = logreg_pipeline.predict(X_val)

print("Отчёт по классам (LogisticRegression):")
print(classification_report(y_val, y_pred_logreg, zero_division=0))

Отчёт по классам (LogisticRegression):
              precision    recall  f1-score   support

       blues       0.38      0.21      0.27       690
     country       0.42      0.54      0.47       817
     hip hop       0.51      0.61      0.55       136
        jazz       0.46      0.32      0.38       577
         pop       0.36      0.50      0.42      1056
      reggae       0.49      0.48      0.49       375
        rock       0.39      0.29      0.33       605

    accuracy                           0.41      4256
   macro avg       0.43      0.42      0.42      4256
weighted avg       0.41      0.41      0.40      4256



## 5. K ближайших соседей (KNN)

In [11]:
# KNN - простая нелинейная модель без настройки.
knn_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5)),
    ]
)

evaluate_model(knn_pipeline, X_train, y_train, X_val, y_val, "KNN (k=5, baseline)")

KNN (k=5, baseline): Macro F1 = 0.3660, Accuracy = 0.3593


{'Модель': 'KNN (k=5, baseline)',
 'Macro F1 (val)': 0.366,
 'Accuracy (val)': 0.3593}

## 6. Таблица результатов

In [14]:
results_df = pd.DataFrame(results).sort_values("Macro F1 (val)", ascending=False)
results_df = results_df.reset_index(drop=True)

print("=" * 61)
print("Сводная таблица базовых моделей:")
print("=" * 61)
print(results_df.to_string(index=False))
print("=" * 61)

Сводная таблица базовых моделей:
                       Модель  Macro F1 (val)  Accuracy (val)
LogisticRegression (baseline)          0.4161          0.4077
          KNN (k=5, baseline)          0.3660          0.3593
 DummyClassifier (stratified)          0.1419          0.1713


## 7. Оценка лучшей базовой модели на тестовой выборке

In [15]:
trained_models = {
    "DummyClassifier (stratified)": dummy,
    "LogisticRegression (baseline)": logreg_pipeline,
    "KNN (k=5, baseline)": knn_pipeline,
}

best_model_name = results_df.iloc[0]["Модель"]
best_model = trained_models[best_model_name]
print(f"Лучшая базовая модель по val: {best_model_name}")

# Переобучаем лучшую модель на полном train, оцениваем на test
best_model.fit(X_train, y_train)
y_pred_test = best_model.predict(X_test)

test_macro_f1 = f1_score(y_test, y_pred_test, average="macro", zero_division=0)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"\nРезультаты {best_model_name} на тестовой выборке:")
print(f"  Macro F1 = {test_macro_f1:.4f}")
print(f"  Accuracy = {test_accuracy:.4f}")

Лучшая базовая модель по val: LogisticRegression (baseline)

Результаты LogisticRegression (baseline) на тестовой выборке:
  Macro F1 = 0.4228
  Accuracy = 0.4088


## 8. Вывод

**Что сделано:**
- Установлена нижняя граница качества (DummyClassifier)
- Обучены две базовые модели без настройки гиперпараметров
- Все модели использовали полный набор из 27 признаков** (6 аудио + 16 тематических + 3 текстовых + 2 мета)

**Итоговые результаты на val:**

| Модель | Macro F1 | Accuracy |
|--------|----------|----------|
| DummyClassifier (stratified) | 0.1419 | 0.1713 |
| KNN (k=5) | 0.3660 | 0.3593 |
| LogisticRegression | 0.4161 | 0.4077 |

**Лучшая модель (LogisticRegression) на тестовой выборке:** Macro F1 = 0.4228, Accuracy = 0.4088

LogisticRegression в **3× лучше** случайного угадывания (0.14) - признаки несут реальную информацию о жанре.

**Наблюдения по классам:**
- hip hop - лучший F1=0.55: жанр появился в 1980-х и имеет отличительные аудио-признаки (высокая энергия, маленький возраст треков)
- blues - худший F1=0.27: сильно пересекается с jazz и rock по аудио-профилю
- country и pop имеют Recall > Precision - модель предсказывает их с избытком, что логично для самых многочисленных классов